# 32. Box scores for franchise mode (engine only, no UI yet)

The franchise screens will have a **month calendar**: every game shows its result, and clicking a game opens its
**box score**. Notebook 30's one-season story has the game log, but player stats only as season totals. This
notebook builds the per-game pieces in `src/boxscore.py`:

1. **Per-game player stats** for a story: each skater's goals, assists, points and minutes in every game, each
   starting goalie's goals against and decision, and who scored and assisted every goal.
2. **Pre-game win probabilities** for every game, from the same game model the simulator uses.
3. **A lineup rule fix:** "moved" now means a player changed **line or pair**, not his order inside it.
4. **A calendar helper:** one row per game for a team and a month.

**What a box score can't show:** periods, goal times, shots, saves, +/- and penalties. The engine doesn't
simulate them, so nothing is invented. Minutes are each player's lineup minutes per game: the engine doesn't vary
ice time from game to game.

This notebook writes nothing.

In [1]:
import sys
sys.path.append("../src")

import subprocess
import time
import types
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import skellam, poisson

import season as SE
import simulate as S
import refresh as R
import franchise as FR
import boxscore as BX

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
BUNDLE_DIR = DATA / "simulator_2026-27"
TRADES = DATA / "trades_2026-27.csv"
CACHE_ROOT = BUNDLE_DIR / "updated"
SIT = list(pd.read_csv(DATA / "sit_out_2026-27.csv")["playerId"])     # Larkin and Edvinsson sit, as in notebook 30
TEAM = "DET"
OPENING = "2026-09-29"
SPEED = {}

meta = R.load_update_cache(R.newest_cache(CACHE_ROOT), SE.load_bundle(BUNDLE_DIR, TRADES), BUNDLE_DIR)[2]
START = meta["start"]
inp = FR.load_inputs(BUNDLE_DIR, TRADES, START, use_update=True, cache_root=CACHE_ROOT)    # notebook 30's inputs
B = inp["bundle"]
print(f"Start {START}, ratings {inp['ratings']}; the same inputs as notebook 30's stories")

Start 2026-11-16, ratings updated through 2026-11-15 (notebook 29); the same inputs as notebook 30's stories


## 1. Splitting a story's season totals into games

**The story already knows:**

- each player's goals and assists for the whole range (notebook 27's season-level split);
- every game's score. The goals players scored in a game = the score minus the shootout winner's +1.

**What it doesn't know:** who scored in which game. `split_story` fills that in.

- **Goals:** each player's goals become "tokens". They're shuffled, then poured into his team's games in order, so
  each game gets exactly its number of goals.
  - Every game's player goals = the team's goals.
  - Every player's games add up to his story total.
  - Both hold by construction.
- **Why that's the same as playing it goal by goal:**
  - Lineups are fixed all season, and every dressed player plays every game with the same minutes. So every goal his
    team scores has the same chance of being his, whichever game it's in.
  - Played goal by goal, each scorer is an independent pick with those chances. Once you know how many goals each
    player ended up with, every way of arranging those goals across the games is equally likely, and a random
    shuffle is exactly that.
  - So the totals and each game's box score come out with the same statistics either way. It's the same argument
    notebook 27 used to split the season at once.
- **Assists:**
  1. **How many each goal gets (0, 1 or 2):** from the same model notebook 27 used for the season (the real 0/1/2
     mix), given the team's total assists. Which goals get 2, 1 or 0 is random.
  2. **Who gets them:** the players' assist tokens fill those slots. Never the scorer, never the same player twice on
     one goal, and each player's total stays exactly his story total.
     - First a random fill, players with the most assists first, up to 20 tries.
     - If it keeps failing, a max-flow solver (a standard method for matching under limits) finds a legal
       arrangement and decides itself which goals get two assists.
  3. **If no legal arrangement exists at all,** the assist that can't be placed goes to the eligible teammate with the
     most assists, so the rules hold and that one assist moves. Section 3 measures how often this happens.
- **Only dressed players, only their games:** scratched and sitting players get nothing. The starting goalie of each
  game gets 60 minutes, his goals against (no shootout or empty-net goals, as in notebook 27) and the decision. A
  backup who didn't start that game has no line in it.
- **Real games before the start** use their real boxscore lines (NHL data). They have no goal list, because the
  boxscores don't pair each goal with its assists.

In [2]:
clock = time.time()
sF = FR.story(inp, TEAM, sit_out=SIT, seed=0)                  # notebook 30's full-season story (seed 0)
t_story = time.time() - clock
clock = time.time()
sp = BX.split_story(sF)
SPEED["per-game split, full season (all 32 teams)"] = time.time() - clock
r = sF["record"]
print(f"{TEAM} from {START}: {r['wins']}-{r['losses']}-{r['otl']} in {r['gp']} games, {r['points']} points "
      f"(notebook 30 printed 29-25-9, 67 points). Split in {SPEED['per-game split, full season (all 32 teams)']:.2f} s")
pg = sp["player_games"]
print(f"Player-game rows: {len(pg):,} ({(pg['kind'] == 'simulated').sum():,} simulated, {(pg['kind'] == 'real').sum():,} real); "
      f"goals listed: {len(sp['goals']):,}")
print("How each team's assists were placed:", pd.Series(sp["assists_how"]).value_counts().to_dict())
pg[(pg["team"] == TEAM) & (pg["kind"] == "simulated")].head(8)

DET from 2026-11-16: 29-25-9 in 63 games, 67 points (notebook 30 printed 29-25-9, 67 points). Split in 0.24 s
Player-game rows: 51,106 (38,532 simulated, 12,574 real); goals listed: 6,179
How each team's assists were placed: {'random': 32}


,game_id,date,team,opponent,home_away,kind,playerId,name,position,g,a,minutes,ga,decision,p
23480,2026020344,2026-11-18,DET,BOS,home,simulated,8482078,Lucas Raymond,L,0,1,19.382310,NaN,NaN,1
23481,2026020344,2026-11-18,DET,BOS,home,simulated,8479337,Alex DeBrincat,R,0,0,18.921823,NaN,NaN,0
23482,2026020344,2026-11-18,DET,BOS,home,simulated,8484471,Emmitt Finnie,C,0,1,16.395573,NaN,NaN,1
23483,2026020344,2026-11-18,DET,BOS,home,simulated,8477429,Andrew Copp,C,1,0,17.010087,NaN,NaN,1
23484,2026020344,2026-11-18,DET,BOS,home,simulated,8478042,Viktor Arvidsson,L,0,0,15.322497,NaN,NaN,0
23485,2026020344,2026-11-18,DET,BOS,home,simulated,8477456,J.T. Compher,L,0,0,15.981871,NaN,NaN,0
23486,2026020344,2026-11-18,DET,BOS,home,simulated,8483464,Marco Kasper,C,0,0,14.368351,NaN,NaN,0
23487,2026020344,2026-11-18,DET,BOS,home,simulated,8478891,Mason Appleton,C,0,0,13.957923,NaN,NaN,0


In [3]:
def split_checks(story, split):
    g = BX.story_games(story)
    pg = split["player_games"]
    sim = pg[pg["kind"] == "simulated"]
    sk = sim[sim["position"] != "G"]
    team_goals = pd.concat([g.set_index(["game_id", "home"])["home_gf"].rename_axis(["game_id", "team"]),
                            g.set_index(["game_id", "away"])["away_gf"].rename_axis(["game_id", "team"])])
    player_sum = sk.groupby(["game_id", "team"])["g"].sum().reindex(team_goals.index).fillna(0)
    rs = story["range_skaters"].set_index("playerId")
    tot = sk.groupby("playerId")[["g", "a"]].sum().reindex(rs.index).fillna(0)
    gl = split["goals"]
    n_ast = gl[["assist1", "assist2"]].notna().sum(axis=1)
    gk = sim[sim["position"] == "G"].groupby("playerId").agg(
        gp=("game_id", "size"), w=("decision", lambda x: (x == "W").sum()), l=("decision", lambda x: (x == "L").sum()),
        otl=("decision", lambda x: (x == "OTL").sum()), ga=("ga", "sum"))
    rg = story["range_goalies"].set_index("playerId")
    lineup = story["run"]["lineups"]
    dressed = set(lineup.loc[lineup["dressed"], "playerId"])
    one_game = sim.groupby(["game_id", "playerId"]).size()
    return {
        f"every team-game: player goals = team goals ({len(team_goals):,} team-games)": bool((player_sum == team_goals).all()),
        "every skater: games add up to his story goals": bool((tot["g"] == rs["sim_g"]).all()),
        "every skater: games add up to his story assists": bool((tot["a"] == rs["sim_a"]).all()),
        "assists: never the scorer": bool(((gl["assist1"] != gl["scorer"]) & (gl["assist2"] != gl["scorer"])).all()),
        "assists: two different players": bool(((gl["assist1"] != gl["assist2"]) | gl["assist2"].isna()).all()),
        "assists: 0 to 2 per goal, filled in order": bool(n_ast.between(0, 2).all() and (gl["assist1"].notna() | gl["assist2"].isna()).all()),
        "goalies: GP, W, L, OTL, GA add up to the story's goalie lines": all(
            (gk[c].reindex(rg.index).fillna(0) == rg[s]).all()
            for c, s in [("gp", "sim_gp"), ("w", "sim_wins"), ("l", "sim_losses"), ("otl", "sim_otl"), ("ga", "sim_ga")]),
        "only dressed players have simulated lines": bool(sim["playerId"].isin(dressed).all()),
        "nobody has two lines in one game": bool((one_game == 1).all()),
        "no assist moved (impossible cases)": len(split["moved"]) == 0,
    }

checks = split_checks(sF, sp)
for k, v in checks.items():
    print(f"{'OK  ' if v else 'FAIL'} {k}")
assert all(checks.values())
gl = sp["goals"]
mix = B["stat_settings"]["mix"]
share = gl[["assist1", "assist2"]].notna().sum(axis=1).value_counts(normalize=True).sort_index()
print(f"\nGoals with 0 / 1 / 2 assists: {share.get(0, 0):.3f} / {share.get(1, 0):.3f} / {share.get(2, 0):.3f} "
      f"(the model's mix: {mix['p0']:.3f} / {mix['p1']:.3f} / {mix['p2']:.3f})")

s10 = FR.story(inp, TEAM, n_games=10, sit_out=SIT, seed=0)    # a short range works the same way
c10 = split_checks(s10, BX.split_story(s10))
print(f"\nNext-10-games story: all {len(c10)} checks {'pass' if all(c10.values()) else 'FAIL'}")
assert all(c10.values())

OK   every team-game: player goals = team goals (2,028 team-games)
OK   every skater: games add up to his story goals
OK   every skater: games add up to his story assists
OK   assists: never the scorer
OK   assists: two different players
OK   assists: 0 to 2 per goal, filled in order
OK   goalies: GP, W, L, OTL, GA add up to the story's goalie lines
OK   only dressed players have simulated lines
OK   nobody has two lines in one game
OK   no assist moved (impossible cases)

Goals with 0 / 1 / 2 assists: 0.062 / 0.182 / 0.756 (the model's mix: 0.065 / 0.184 / 0.751)

Next-10-games story: all 10 checks pass


## 2. Box scores: one simulated game, one real game

`box_score(split, game_id)` is what the click shows. A simulated game has the goal list in order (no times: the
engine has none). A real game has the real player lines and no goal list.

In [4]:
games = sp["games"]
det_sim = games[(games["kind"] == "simulated") & ((games["home"] == TEAM) | (games["away"] == TEAM))]
b = BX.box_score(sp, det_sim["game_id"].iloc[0])
print(f"{b['date']} ({b['kind']}): {b['score']}")
print("\nGoals:")
print(b["goals"].to_string(index=False))
print("\nGoalies:")
print(b["goalies"].to_string(index=False))
print("\nSkaters with a point, and the top minutes:")
sk = b["skaters"]
print(sk[(sk["p"] > 0) | (sk["minutes"] >= sk["minutes"].quantile(0.85))].round(1).to_string(index=False))
print("\n" + b["note"])

2026-11-18 (simulated): BOS 2 at DET 1 (OT)

Goals:
team  goal_no        scorer_name   assist1_name   assist2_name
 BOS        1 Marat Khusnutdinov    Pavel Zacha  Mark Kastelic
 BOS        2       Mason Lohrei Charlie McAvoy Elias Lindholm
 DET        1        Andrew Copp  Lucas Raymond  Emmitt Finnie

Goalies:
team             name  ga decision  minutes
 BOS Michael DiPietro 1.0        W     60.0
 DET      John Gibson 2.0      OTL     60.0

Skaters with a point, and the top minutes:
team               name position  g  a  p  minutes
 BOS       Mason Lohrei        D  1  0  1     17.1
 BOS Marat Khusnutdinov        C  1  0  1     14.1
 BOS     Charlie McAvoy        D  0  1  1     23.8
 BOS     Elias Lindholm        C  0  1  1     17.1
 BOS        Pavel Zacha        C  0  1  1     16.5
 BOS      Mark Kastelic        C  0  1  1     12.0
 BOS    Hampus Lindholm        D  0  0  0     21.3
 BOS     Nikita Zadorov        D  0  0  0     20.4
 DET        Andrew Copp        C  1  0  1     17.0


In [5]:
det_real = games[(games["kind"] == "real") & ((games["home"] == TEAM) | (games["away"] == TEAM))]
br = BX.box_score(sp, det_real["game_id"].iloc[-1])
print(f"{br['date']} ({br['kind']}): {br['score']}; goal list rows: {len(br['goals'])} (real boxscores don't pair goals and assists)")
print(br["goalies"].to_string(index=False))
print(br["skaters"][br["skaters"]["p"] > 0].round(1).to_string(index=False))

2026-11-14 (real): DET 2 at TBL 0; goal list rows: 0 (real boxscores don't pair goals and assists)
team           name  ga decision  minutes
 DET     D. Tarasov 0.0        W     60.0
 TBL A. Vasilevskiy 1.0        L     57.9
team                name position  g  a  p  minutes
 DET          L. Raymond        L  1  1  2     13.6
 DET M. Brandsegg-Nygård        R  1  0  1     14.6
 DET             A. Copp        C  0  1  1     17.0
 DET        W. Wallinder        D  0  1  1     16.8


## 3. How often are the assist rules impossible?

A legal arrangement can only fail to exist when a player has more assists than the goals his team scored without
him (he can assist each of those at most once), or in rare crowded cases the flow solver finds. Two measurements:

- **10,000 full seasons** from opening night: the per-player condition, on every team-season (from the season totals,
  `run_scenario(keep_seasons=True)`).
- **200 full stories** split game by game: how each team's assists were placed (random fill, flow solver, impossible).

In [6]:
inp_open = FR.load_inputs(BUNDLE_DIR, TRADES, OPENING)
res = SE.run_scenario(inp_open["bundle"], n_sims=10000, seed=11, stat_lines=True, keep_seasons=True)
team_of = res["skaters"]["team"].to_numpy()
teams = np.unique(team_of)
G = res["season_goals"].astype(int)
A = res["season_assists"].astype(int)
team_goals = np.stack([G[:, team_of == t].sum(1) for t in teams], axis=1)
room = team_goals[:, np.searchsorted(teams, team_of)] - G             # goals his team scored without him
over = A > room
print(f"10,000 seasons x 32 teams = {G.shape[0] * len(teams):,} team-seasons; player-seasons where assists > goals "
      f"scored without him: {int(over.sum())}")
print(f"Closest case: a player's assists were {int((room - A).min())} below that limit "
      f"(most assists by one player in one season: {int(A.max())}; fewest team goals: {int(team_goals.min())})")

clock = time.time()
how = []
for seed in range(200):
    st = FR.story(inp_open, TEAM, seed=seed)
    spl = BX.split_story(st, seed=seed)
    how += list(spl["assists_how"].values())
    assert all(split_checks(st, spl).values())
how = pd.Series(how).value_counts()
print(f"\n200 full stories ({time.time() - clock:.0f} s), {how.sum():,} team-seasons, every check passing in each: "
      + ", ".join(f"{k} {v:,}" for k, v in how.items()))

10,000 seasons x 32 teams = 320,000 team-seasons; player-seasons where assists > goals scored without him: 0
Closest case: a player's assists were 49 below that limit (most assists by one player in one season: 153; fewest team goals: 130)

200 full stories (160 s), 6,400 team-seasons, every check passing in each: random 6,400


**The impossible case never happened:**

- **In 10,000 full seasons** (320,000 team-seasons), no player ever had more assists than the goals his team scored
  without him.
  - The closest anyone came was 49 assists below that limit.
  - The most assists by one player in any of those seasons was 153, an extreme tail, and the fewest goals by a team
    was 130.
- **In 200 full stories split game by game,** the random fill placed every team's assists within its 20 tries, all
  6,400 times. The flow solver was never needed, and no assist ever had to move.
- **Both fallbacks stay in the code as a safety net,** for odd custom setups such as very short ranges or unusual
  lineups.

## 4. Pre-game win probabilities

`simulate.game_probabilities(edge, settings)` gives the **exact** chances for one game, with no simulation, from the
same pieces `simulate_games` uses:

- the strength edge (skater and goalie WAR per game, plus the team bonus) sets each team's expected normal goals;
- the normal-goal margin follows the Skellam distribution (the difference of two Poisson counts);
- the late-game table adds empty-net and pulled-goalie goals by margin;
- a tie after regulation goes to OT/SO, won by the home team with the fitted logistic chance.

**Check 1:** the formula vs. 400,000 simulated games at a few edges.

In [7]:
st = B["game"]
rows = []
for i, e in enumerate([-0.6, -0.2, 0.0, 0.3, 0.8]):
    p = S.game_probabilities(e, st)
    n = 400000
    hg, ag, ot, hw = S.simulate_games(np.array([0]), np.array([1]), np.tile([[e / st["a_s"], 0.0]], (n, 1)),
                                      np.zeros((n, 2)), st, np.random.default_rng([32, i]))
    rows.append({"edge": e, "home win (exact)": p["home_win"][0], "home win (simulated)": hw.mean(),
                 "OT (exact)": p["ot"][0], "OT (simulated)": ot.mean(),
                 "2 SE": 2 * np.sqrt(p["home_win"][0] * (1 - p["home_win"][0]) / n)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

diff = st["home"] + np.array([-0.6, 0.0, 0.8])
mh, ma = np.clip((st["total"] + diff) / 2, 0.05, None), np.clip((st["total"] - diff) / 2, 0.05, None)
ms = np.arange(-20, 21)
k = np.arange(61)
ph, pa = poisson.pmf(k[None, :], mh[:, None]), poisson.pmf(k[None, :], ma[:, None])
built = np.array([[(ph[i, m:] * pa[i, :61 - m]).sum() if m >= 0 else (ph[i, :61 + m] * pa[i, -m:]).sum() for m in ms]
                  for i in range(3)])
gap_sk = np.abs(built - skellam.pmf(ms[None, :], mh[:, None], ma[:, None])).max()
grid, pgrid = BX.prob_grid(st)
x = np.random.default_rng(0).uniform(-2, 2, 2000)
ex, lk = S.game_probabilities(x, st), BX.lookup(x, grid, pgrid)
print(f"\nMargin chances built from two Poisson counts vs. scipy's skellam: largest gap {gap_sk:.1e}. "
      f"The lookup grid ({len(grid):,} edges) vs. the exact formula: largest gap "
      f"{max(np.abs(ex[k] - lk[k]).max() for k in BX.PROB_KEYS):.1e}")

 edge  home win (exact)  home win (simulated)  OT (exact)  OT (simulated)   2 SE
 -0.6            0.4314                0.4326      0.2188          0.2196 0.0016
 -0.2            0.5042                0.5030      0.2211          0.2216 0.0016
  0.0            0.5407                0.5412      0.2202          0.2199 0.0016
  0.3            0.5948                0.5949      0.2163          0.2169 0.0016
  0.8            0.6810                0.6810      0.2034          0.2038 0.0015

Margin chances built from two Poisson counts vs. scipy's skellam: largest gap 2.2e-16. The lookup grid (32,001 edges) vs. the exact formula: largest gap 1.6e-09


**For every game in the range,** with the lineups in effect (franchise choices, each game's starting goalie, the
team bonus, and the in-season update when it's on). The main number averages the chances over the players' random
seasons (the same 10,000 draws as the projection with this seed): before a game, nobody knows how each player's
season is going, so that's the honest pre-game number from the model. "At projection" is the same with every
player exactly at his projection, for comparison.

**Check 2:** add up each team's chances over the range and compare with the same 10,000 simulated seasons: its
average wins and its average number of games that went past regulation. Both use the same player seasons, so only
the game-to-game luck differs. The standard error pairs each simulated season with its own expected wins, so it
measures exactly that luck. They should agree within about 2 standard errors, and the squared gaps should add up to
about 32 (one per team) if the difference is only noise.

In [8]:
clock = time.time()
wp = BX.win_probabilities(inp, TEAM, sit_out=SIT, seed=0)
SPEED["win probabilities, every game in the range (incl. its 10,000-season run)"] = time.time() - clock
chk = BX.check_against_runs(wp)
print(f"{len(wp['games']):,} games in {SPEED['win probabilities, every game in the range (incl. its 10,000-season run)']:.1f} s")
print(chk[["team", "games", "expected_wins", "sim_wins", "se_wins", "wins_gap_in_se", "expected_ot", "sim_ot",
           "ot_gap_in_se"]].round(3).to_string(index=False))
print(f"\nWins: largest gap {chk['wins_gap_in_se'].abs().max():.2f} SE, teams beyond 2 SE: {(chk['wins_gap_in_se'].abs() > 2).sum()}; "
      f"OT games: largest {chk['ot_gap_in_se'].abs().max():.2f} SE, beyond 2 SE: {(chk['ot_gap_in_se'].abs() > 2).sum()} of 32 "
      f"(about 1.6 expected by chance)")
print(f"Sum of squared gaps (about 32 expected if it's only noise): wins {(chk['wins_gap_in_se'] ** 2).sum():.1f}, "
      f"OT {(chk['ot_gap_in_se'] ** 2).sum():.1f}")

1,014 games in 8.2 s
team  games  expected_wins  sim_wins  se_wins  wins_gap_in_se  expected_ot  sim_ot  ot_gap_in_se
 ANA     63         35.441    35.440    0.038           0.039       13.012  13.011         0.020
 BOS     64         36.087    36.032    0.038           1.472       13.201  13.221        -0.622
 BUF     64         32.829    32.806    0.038           0.618       13.385  13.378         0.208
 CAR     63         41.592    41.590    0.036           0.063       12.373  12.327         1.450
 CBJ     66         26.800    26.828    0.038          -0.760       13.448  13.389         1.829
 CGY     63         24.645    24.683    0.037          -1.023       12.776  12.746         0.960
 CHI     64         22.606    22.628    0.037          -0.591       12.663  12.619         1.393
 COL     63         40.690    40.691    0.036          -0.023       12.480  12.444         1.117
 DAL     64         35.557    35.592    0.038          -0.909       13.210  13.270        -1.863
 DET     

In [9]:
g = wp["games"]
d = g["home_win"] - g["home_win_at_proj"]
print(f"With the update on: averaged vs. at projection, home win % differs by {d.abs().mean() * 100:.1f} points on average "
      f"(largest {d.abs().max() * 100:.1f}). OT %: {g['ot'].mean() * 100:.1f} averaged vs. {g['ot_at_proj'].mean() * 100:.1f} at projection")
wp_open = BX.win_probabilities(inp_open, TEAM, seed=0)
go = wp_open["games"]
do = go["home_win"] - go["home_win_at_proj"]
print(f"Preseason, from opening night: they differ by {do.abs().mean() * 100:.2f} points on average (largest {do.abs().max() * 100:.2f})")
print(f"Home teams win {go['home_win'].mean() * 100:.1f}% on average; {go['ot'].mean() * 100:.1f}% of games go past regulation")

With the update on: averaged vs. at projection, home win % differs by 3.8 points on average (largest 12.7). OT %: 20.5 averaged vs. 21.4 at projection
Preseason, from opening night: they differ by 0.32 points on average (largest 1.23)
Home teams win 54.0% on average; 20.9% of games go past regulation


**What the probabilities show:**

- **The formula is exact.**
  - It matches 400,000 simulated games at every edge tested, within 2 standard errors.
  - The margin chances built from two Poisson counts equal scipy's Skellam to 2e-16.
  - The fast lookup grid is within 2e-9 of the exact formula.
- **It agrees with the 10,000 seasons.**
  - **Wins:** every team's summed chances are within 1.6 standard errors of its simulated average. The squared
    gaps add up to 22.5, close to the 32 expected from noise alone.
  - **Games past regulation:** two teams are just beyond 2 standard errors (Detroit +2.3, Los Angeles -2.5), about
    what chance gives in 32 teams. The squared gaps add up to 40.0, within the normal range for a sum of 32
    (about ± 8).
- **Averaged vs. "at projection":**
  - **Preseason (opening night):** the two differ by only 0.32 points of win % on average (at most 1.2). Players'
    ups and downs mostly cancel out within a game.
  - **With the in-season update on:** they differ by 3.8 points on average, up to 12.7. The update has moved some
    players' draws away from 0 (their real hot and cold starts), and "at projection" ignores that.
  - **So the averaged number is the right one for the calendar.**
- **The league as a whole:** home teams win 54.0% of games, and 20.9% go past regulation.

## 5. The calendar

`calendar(games, team, month, probs)`: one row per game of the team in that month.

- **Story mode** (the split's games): the result and the record after it. The record counts the real games before the
  start, so it's the season record.
- **Projection mode** (the season's games + win probabilities): win %, OT % and loss % for every game still to play.

In both modes, real games before the start show their real result.

In [10]:
clock = time.time()
cal_story = BX.calendar(sp["games"], TEAM, "2026-11")
SPEED["one month of calendar, story mode"] = time.time() - clock
print(f"{TEAM}, November 2026, story mode (real games through {meta['evidence_through']}, then the story):")
print(cal_story.to_string(index=False))
clock = time.time()
cal_proj = BX.calendar(BX.season_games(B, START), TEAM, "2026-12", probs=wp["games"])
SPEED["one month of calendar, projection mode (given the probabilities)"] = time.time() - clock
print(f"\n{TEAM}, December 2026, projection mode:")
print(cal_proj.round(1).to_string(index=False))

DET, November 2026, story mode (real games through 2026-11-15, then the story):
      date opponent home_away      kind    game_id       result  record
2026-11-02      FLA      away      real 2026020233        W 5-3   7-5-3
2026-11-03      TBL      away      real 2026020238        L 2-4   7-6-3
2026-11-05      VGK      home      real 2026020252        L 1-2   7-7-3
2026-11-07      COL      away      real 2026020274        L 0-4   7-8-3
2026-11-10      STL      away      real 2026020290        L 2-3   7-9-3
2026-11-12      FLA      away      real 2026020302        L 4-5  7-10-3
2026-11-14      TBL      away      real 2026020316        W 2-0  8-10-3
2026-11-18      BOS      home simulated 2026020344 OTL 1-2 (OT)  8-10-4
2026-11-21      NYI      home simulated 2026020367        L 3-4  8-11-4
2026-11-25      VAN      home simulated 2026020394        L 1-3  8-12-4
2026-11-27      CBJ      away simulated 2026020407        W 3-1  9-12-4
2026-11-28      NSH      home simulated 2026020417   W 2

## 6. The lineup rule fix: "moved" = changed line or pair

**Before:** a player was "moved" if his slot changed, so swapping two wingers inside line 1 changed their minutes.

**Now:** a player is moved only if he changed **line** (F1-F3 = line 1, ..., F10-F12 = line 4) or **pair** (D1-D2,
D3-D4, D5-D6), or wasn't dressed before.

- **His minutes follow a "minutes rank":** a player who stays in his line keeps his baseline rank, and moved
  players take the line's remaining ranks in slot order.
  - Every slot's minutes are still used exactly once, so the forward and defense totals don't change.
  - The order you set (F1, F2, ...) is still what's displayed.
- **It applies under all three minutes rules,** so reordering inside a line never changes anyone's minutes.

The checks below compare with the committed version of franchise.py (loaded from git, in memory).

In [11]:
def head_module(name, deps):
    """The committed (HEAD) version of src/<name>.py, loaded in memory (no files written)."""
    src = subprocess.run(["git", "show", f"HEAD:src/{name}.py"], capture_output=True, text=True, check=True).stdout
    mod = types.ModuleType(f"{name}_head")
    saved = {k: sys.modules.get(k) for k in deps}
    sys.modules.update(deps)
    try:
        exec(compile(src, f"HEAD:src/{name}.py", "exec"), mod.__dict__)
    finally:
        for k, v in saved.items():
            sys.modules[k] = v
    return mod

SE_head = head_module("season", {})
FR_head = head_module("franchise", {"season": SE_head})
base = FR.base_lineups(inp, sit_out=SIT)
det = FR.lineup_table(base, TEAM)
pid = lambda name: int(B["players"].loc[B["players"]["name"] == name, "playerId"].iloc[0])
mins = lambda lu: lu["minutes_per_game"].to_numpy()

wing = FR.swap_slots(det, TEAM, "F1", "F2")
print("Winger swap on line 1:", " <-> ".join(det.loc[det["slot"].isin(["F1", "F2"]), "name"]))
rows = []
for rule in FR.MINUTES_RULES:
    same_rule = FR.apply_overrides(base, B, det, minutes_rule=rule)[0]          # the same rule, nothing changed
    new = FR.apply_overrides(base, B, wing, minutes_rule=rule)[0]
    old = FR_head.apply_overrides(base, B, wing, minutes_rule=rule)[0]
    rows.append({"rule": rule, "now: same minutes as no change": np.array_equal(mins(new), mins(same_rule)),
                 "before: players whose minutes changed": int((mins(old) != mins(same_rule)).sum())})
pair = FR.swap_slots(det, TEAM, "D1", "D2")
print(pd.DataFrame(rows).to_string(index=False))
print("D pair swap (D1 <-> D2): same minutes as no change:", np.array_equal(mins(FR.apply_overrides(base, B, pair)[0]), mins(base)))

cases = {"(a) lines 1 and 4 swapped": FR.swap_lines(det, TEAM, "F", 1, 4),
         "(b) Bear for Mazur (F12)": FR.replace(det, TEAM, pid("Carter Mazur"), pid("Carter Bear"), base),
         "G1/G2 swap": FR.swap_slots(det, TEAM, "G1", "G2")}
print("\nNotebook 30's lineup changes, minutes now vs. the committed version:")
for name, o in cases.items():
    same = all(np.array_equal(mins(FR.apply_overrides(base, B, o, minutes_rule=r)[0]),
                              mins(FR_head.apply_overrides(base, B, o, minutes_rule=r)[0])) for r in FR.MINUTES_RULES)
    print(f"  {name}: identical under all three rules: {same}")

Winger swap on line 1: Lucas Raymond <-> Alex DeBrincat
 rule  now: same minutes as no change  before: players whose minutes changed
moved                            True                                     12
blend                            True                                      2
 slot                            True                                      2
D pair swap (D1 <-> D2): same minutes as no change: True

Notebook 30's lineup changes, minutes now vs. the committed version:
  (a) lines 1 and 4 swapped: identical under all three rules: True
  (b) Bear for Mazur (F12): identical under all three rules: True
  G1/G2 swap: identical under all three rules: True


In [12]:
mixed = FR.swap_slots(FR.swap_slots(det, TEAM, "F1", "F3"), TEAM, "F2", "F11")
lu = FR.apply_overrides(base, B, mixed)[0]
d = lu[(lu["team"] == TEAM) & lu["dressed"] & (lu["group"] == "F")].set_index("playerId")
b0 = base[(base["team"] == TEAM) & base["dressed"] & (base["group"] == "F")].set_index("playerId")
show = pd.DataFrame({"slot": d["slot"], "name": d["name"], "baseline slot": b0["slot"].reindex(d.index),
                     "baseline minutes": b0["minutes_per_game"].reindex(d.index), "minutes now": d["minutes_per_game"]})
show = show.sort_values("slot", key=lambda s: s.map({x: i for i, x in enumerate(FR.SLOTS)}))
print("A mixed example: F1 <-> F3 (same line) and F2 <-> F11 (lines 1 and 4):")
print(show.round(2).to_string(index=False))
print(f"Forward minutes: {d['minutes_per_game'].sum():.4f} now, {b0['minutes_per_game'].sum():.4f} baseline")

pF = FR.projection(inp, TEAM, sit_out=SIT)
p_wing = FR.projection(inp, TEAM, sit_out=SIT, overrides=wing)
p_swap = FR.projection(inp, TEAM, sit_out=SIT, overrides=cases["(a) lines 1 and 4 swapped"])
dw, ds = FR.paired_change(pF, p_wing, TEAM), FR.paired_change(pF, p_swap, TEAM)
print(f"\n10,000 seasons (same seed): winger swap {dw['points']:+.2f} points (every season's points identical: "
      f"{np.array_equal(p_wing['run']['points'], pF['run']['points'])}); "
      f"lines 1 and 4 swapped {ds['points']:+.2f} ± {ds['points_2se']:.2f} (notebook 30: -2.35 ± 0.20)")

A mixed example: F1 <-> F3 (same line) and F2 <-> F11 (lines 1 and 4):
slot                     name baseline slot  baseline minutes  minutes now
  F1            Emmitt Finnie            F3             16.40        16.38
  F2             John Leonard           F11             11.88        18.81
  F3            Lucas Raymond            F1             19.38        19.36
  F4              Andrew Copp            F4             17.01        16.99
  F5         Viktor Arvidsson            F5             15.32        15.31
  F6             J.T. Compher            F6             15.98        15.96
  F7             Marco Kasper            F7             14.37        14.35
  F8           Mason Appleton            F8             13.96        13.94
  F9        Michael Rasmussen            F9             13.10        13.09
 F10 Michael Brandsegg-Nygård           F10             12.78        12.77
 F11           Alex DeBrincat            F2             18.92        12.16
 F12             Carter Mazur

## 7. Nothing else changed

- **season.py** only gained two kept arrays (each game's empty-net goals, when a story keeps its games). Every
  default run is identical to the committed version.
- **franchise.py with no overrides** still equals `run_scenario`, bit for bit, as in notebook 30.

In [13]:
def same(x, y):
    if isinstance(x, dict):
        return set(x) == set(y) and all(same(x[k], y[k]) for k in x)
    if isinstance(x, pd.DataFrame):
        return x.equals(y)
    return np.array_equal(x, y, equal_nan=True)

KEYS = ["table", "tallies", "seeds", "points", "skaters", "goalies", "lineups"]
rows = []
for n, seed in [(1, 0), (10000, 7)]:                  # notebook 30's pairs: a one-season story at seed 0
    a = SE_head.run_scenario(B, start_date=START, sit_out=SIT, n_sims=n, seed=seed, stat_lines=True, updated=inp["updated"])
    b = SE.run_scenario(B, start_date=START, sit_out=SIT, n_sims=n, seed=seed, stat_lines=True, updated=inp["updated"])
    m = FR.play(inp, TEAM, sit_out=SIT, n_sims=n, seed=seed, story=(n == 1))
    rows.append({"seasons": n, "seed": seed, "season.py committed = edited": all(same(a[k], b[k]) for k in KEYS),
                 "franchise (no overrides) = run_scenario": all(same(b[k], m[k]) for k in KEYS)})
print(pd.DataFrame(rows).to_string(index=False))
old = FR_head.story(inp, TEAM, sit_out=SIT, seed=0)["run"]["games"]
new = sF["run"]["games"]
print("Story games: every committed array identical:", all(np.array_equal(old[k], new[k]) for k in old),
      "| added:", sorted(set(new) - set(old)))

 seasons  seed  season.py committed = edited  franchise (no overrides) = run_scenario
       1     0                          True                                     True
   10000     7                          True                                     True
Story games: every committed array identical: True | added: ['away_en', 'home_en']


## 8. Speed and git

In [14]:
print("Speed (seconds):")
print(f"  one full-season story (notebook 30): {t_story:.2f}")
for k, v in SPEED.items():
    print(f"  {k}: {v:.2f}")

def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd="..").stdout.rstrip()

print("\n" + git("check-ignore", "-v", "data/box_scores_2026-27.csv"))
tracked = git("ls-files", "data").split("\n")
print("Tracked files under data/:", tracked)
assert tracked == ["data/README.md"]
print("\ngit status --porcelain:")
print(git("status", "--porcelain"))

Speed (seconds):
  one full-season story (notebook 30): 0.64
  per-game split, full season (all 32 teams): 0.24
  win probabilities, every game in the range (incl. its 10,000-season run): 8.19
  one month of calendar, story mode: 0.01
  one month of calendar, projection mode (given the probabilities): 0.01

.gitignore:2:data/*.csv	data/box_scores_2026-27.csv
Tracked files under data/: ['data/README.md']

git status --porcelain:
 M CLAUDE.md
 M src/franchise.py
 M src/season.py
 M src/simulate.py
?? notebooks/32_box_scores.ipynb
?? src/boxscore.py


## Takeaways

- **Every story now has box scores.**
  - **Coverage:** `split_story` gives every dressed player a line in every game of the range: goals, assists,
    points, minutes, and each starting goalie's goals against and decision. It also lists who scored and assisted
    every goal.
  - **The full season splits in 0.2 s,** with notebook 30's story numbers unchanged (Detroit 29-25-9, 67 points).
  - **All ten checks pass:**
    - every one of the 2,028 team-games has player goals equal to team goals;
    - every player's games add up to his story goals and assists;
    - assists are never the scorer's and never two from one player;
    - goalie lines add up to the story's;
    - only dressed players appear, once per game.
- **Why it's the same as playing goal by goal:** with fixed lineups every goal has the same chances in every game,
  so once each player's total is known, any arrangement across the games is equally likely. The 0/1/2 assist mix
  comes out as the model says: 6.2% / 18.2% / 75.6% against 6.5% / 18.4% / 75.1%.
- **The assist rules were never impossible:** 0 cases in 10,000 seasons, and in 200 game-by-game stories the
  random fill always worked. The flow solver and the leftover rule are kept as a safety net.
- **A box score shows** the score, both teams' skaters, the goalies, and the goal list in order. Real games before
  the start show their real lines, without a goal list. No periods, goal times, shots, saves, +/- or penalties: the
  engine doesn't simulate them.
- **Pre-game win probabilities are exact, from the simulator's own game model.**
  - **Matches the simulator:** they equal simulated games, and summed over the range they agree with the 10,000
    seasons (wins within 1.6 standard errors for every team).
  - **The calendar uses the chance averaged over player seasons.** It matters once the in-season update is on
    (3.8 points of win % on average) and barely before it (0.3).
  - **Speed:** every game in the range takes 7.4 s, including the 10,000-season run.
- **The calendar helper** gives a team's month: results and the running season record in story mode (real games
  first), and win / OT / loss chances in projection mode. One month takes about 0.01 s.
- **The lineup rule fix works.**
  - **Reordering inside a line or pair changes nothing:** under all three minutes rules, swapping two wingers on
    line 1 gives exactly the no-change minutes. Before, it changed 12 players' minutes under the default rule. The
    10,000-season points are identical.
  - **Notebook 30's changes are unchanged:** the line 1/4 swap (-2.35 ± 0.20 points), Bear for Mazur and the
    goalie swap give exactly the committed minutes.
  - **Players who change lines still take the new line's minutes,** and the team totals stay the same (180.62
    forward minutes per game).
- **Nothing else changed:** season.py's default runs and franchise mode with no overrides are identical, bit for
  bit, to the committed versions. A story only gained each game's empty-net goals.
- **Files:**
  - **New:** `src/boxscore.py`.
  - **Add-only:** `season.run_scenario` (empty-net goals per game) and `simulate.game_probabilities`.
  - **Changed as requested:** the "moved" rule in `franchise.apply_overrides`.
  - **Nothing is written.** The reserved path data/box_scores_2026-27.csv is ignored by git, and only
    data/README.md is tracked under data/.